In [ ]:
# Config — change MODEL_ID to "vibevoice/VibeVoice-7B" for the larger model
MODEL_ID = "vibevoice/VibeVoice-1.5B"
SPEAKER_NAMES = ["Alice", "Maya"] # chose demo voices based on the order they appear in the script (e.g. Alice will be Speaker 1)
TXT_PATH = "podcast.txt"
OUT_DIR = "outputs"

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none — switch Runtime to GPU (A100 recommended for 7B)")

In [ ]:
!if [ ! -d "VibeVoice" ]; then git clone https://github.com/vibevoice-community/VibeVoice.git; else echo "VibeVoice already cloned."; fi

In [ ]:
!pip install -q -e VibeVoice/
!pip install -q soundfile

In [ ]:
# Paste your VibeVoice clipped script below between the triple quotes
script = """
Speaker 1: Okay so imagine a fire alarm goes off—right? But the building’s also… rewriting itself.
Speaker 2: That’s not ominous at all.
Speaker 1: In Lost, one “must-act-now” choice is the spark. And we’re asking—was Jack saving everyone… or forcing the island to agree with him?
Speaker 2: Hey Maya—welcome back.
"""

In [ ]:
import os, subprocess
os.makedirs(OUT_DIR, exist_ok=True)

with open(TXT_PATH, "w") as f:
    f.write(script)

cmd = [
    "python", "demo/inference_from_file.py",
    "--model_path", MODEL_ID,
    "--txt_path", f"../{TXT_PATH}",
    "--speaker_names", *SPEAKER_NAMES,
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd="VibeVoice", check=True)

In [ ]:
import glob
from IPython.display import Audio, display

wavs = sorted(glob.glob("VibeVoice/outputs/**/*.wav", recursive=True), key=os.path.getmtime)
if not wavs:
    wavs = sorted(glob.glob("VibeVoice/**/*.wav", recursive=True), key=os.path.getmtime)
print("Found:", wavs[-1] if wavs else "no wav files")
if wavs:
    display(Audio(wavs[-1]))